In [1]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

test_df = pd.read_parquet("../data/processed/test_preds.parquet")
ft = pd.read_parquet("../data/processed/ft_preds.parquet")

test_df = test_df.merge(ft, on="review_id")
test_df.to_parquet("../data/processed/test_preds.parquet", index=False)

y = test_df["sentiment"]
print(len(test_df))                      # must be 10000
print(accuracy_score(y, test_df["pred_ft"]))
print(f1_score(y, test_df["pred_ft"], average="macro"))
print(classification_report(y, test_df["pred_ft"]))
print(confusion_matrix(y, test_df["pred_ft"], labels=["negative", "positive"]))

10000
0.9468
0.9467886574702263
              precision    recall  f1-score   support

    negative       0.95      0.94      0.95      4934
    positive       0.95      0.95      0.95      5066

    accuracy                           0.95     10000
   macro avg       0.95      0.95      0.95     10000
weighted avg       0.95      0.95      0.95     10000

[[4661  273]
 [ 259 4807]]


In [ ]:
# === Phase 5: Sentiment model comparison ===
import os, sys
if '..' not in sys.path:
    sys.path.insert(0, '..')
import pandas as pd
from src.sentiment import (
    compute_sentiment_metrics,
    compute_tfidf_bert_overlap,
    plot_sentiment_comparison,
)

test_df = pd.read_parquet("../data/processed/test_preds.parquet")

# 1. Compute comparison metrics
metrics_df = compute_sentiment_metrics(test_df)
os.makedirs("../results", exist_ok=True)
metrics_df.to_csv("../results/sentiment_comparison.csv", index=False)
print("Sentiment Comparison:")
print(metrics_df.to_string(index=False))

# 2. Four-way TF-IDF vs DistilBERT overlap
overlap = compute_tfidf_bert_overlap(test_df)
print("\nFour-way TF-IDF vs DistilBERT Correctness Overlap:")
for k, v in overlap["correctness"].items():
    print(f"  {k:30s}: {v['count']:5d} ({v['pct']}%) ")

print("\nFour-way TF-IDF vs DistilBERT Prediction Agreement:")
for k, v in overlap["agreement"].items():
    print(f"  {k:30s}: {v['count']:5d} ({v['pct']}%) ")

# 3. Plot comparison
plot_sentiment_comparison(metrics_df, output_path="../results/sentiment_comparison.png")
